# The Arithmetic Kakeya Conjecture

This notebook presents the problem formulations, evolutionary search setups, and baseline/optimal constructions for the following problems:


## 30. The Arithmetic Kakeya Conjecture

### Detailed Problem Description
For each slope $r \in \mathbb{R} \cup \{\infty\}$ define the projection $\pi_r : \mathbb{R}^2 \to \mathbb{R}$ by $\pi_r(a,b) = a + rb$ for $r \neq \infty$ and $\pi_\infty(a,b)=b$.  Given a set $r_1,\dots,r_k, r_\infty$ of distinct slopes, we let $C(\{r_1,\dots,r_k\}; r_\infty)$ be the smallest constant for which the following is true: if $X,Y$ are discrete random variables (not necessarily independent) taking values in a finite set of reals, then
$$ {\mathbf H}(\pi_{r_\infty}(X,Y)) \leq C(\{r_1,\dots,r_k\}; r_\infty) \max_{i=1,\dots,k} {\mathbf H}(\pi_{r_i}(X,Y)),$$
where $ {\mathbf H}(X) = -\sum_{x} P(X = x) \log(P(X=x))$ is the entropy of a random variable and $x$ ranges over the values taken by $X$. The <em>arithmetic Kakeya conjecture</em> asserts that $C(\{r_1,\dots,r_k\}; r_\infty)$ can be made arbitrarily close to $1$.


## AlphaEvolve Search Configuration

**Prompt**

Arithmetic Kakeya conjecture

Act as a research mathematician and optimization specialist.

GOAL:
For a given set of slopes, your task is to find a joint probability distribution of discrete random variables X, Y that maximizes the entropy ratio for projections along slopes r_1, ..., r_k vs r_infinity.

Specifically, the Python function you have to provide has the following
signature:

def get_joint_distribution(n: int) -> np.ndarray

EVALUATION:

Your construction will be scored by a function called
get_score.
The interface of get_score is:

def get_score(construction) -> float

Your list of elements will be evaluated by get_score which outputs the entropy projection ratio.
You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have access to it,
you don't need to code up the get_score() function.
You want the score it gives you to be as large as possible!

Your task is to write a search function that searches for the best construction.
Your function will have 1000 seconds to run, and after that it has to have
returned the best construction it found. If after 1000 seconds it has not
returned anything, it will be terminated with negative infinity points. You can
use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to define
the "start_time" variable early in your program.


### Initial Program (Baseline/Search Seed)

In [ ]:
import numpy as np

def initial_distribution(n: int) -> np.ndarray:
    return np.ones((n, n)) / (n * n)

### Evolved Code by AlphaEvolve

In [ ]:
def evolved_discrete_gaussian_distribution(n: int) -> np.ndarray:
    # Joint distribution resembling discrete Gaussians
    x = np.arange(-n//2, n//2 + n%2)
    y = np.arange(-n//2, n//2 + n%2)
    xx, yy = np.meshgrid(x, y)
    P = np.exp(-0.25 * (xx**2 + yy**2))
    return P / np.sum(P)

### Evaluator Function

In [ ]:
from scipy.stats import entropy

def evaluate_kakeya_entropy(P_xy: np.ndarray, slopes: list[float], r_inf: float) -> float:
    ny, nx = P_xy.shape
    def get_projection_dist(r):
        proj_probs = {}
        for y in range(ny):
            for x in range(nx):
                val = x + r * y if r != float('inf') else y
                proj_probs[val] = proj_probs.get(val, 0.0) + P_xy[y, x]
        return np.array(list(proj_probs.values()))
    H_inf = entropy(get_projection_dist(r_inf), base=2)
    max_H_i = 0.0
    for r in slopes:
        H_i = entropy(get_projection_dist(r), base=2)
        if H_i > max_H_i:
            max_H_i = H_i
    return H_inf / max_H_i

### Data Verification and Results

In [ ]:
P = evolved_discrete_gaussian_distribution(10)
slopes = [0.0, 1.0, 2.0]
r_inf = -1.0
print("Entropy ratio for discrete Gaussian:", evaluate_kakeya_entropy(P, slopes, r_inf))